In [0]:
%pip install mlflow databricks-sdk delta-spark pandas pyarrow
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.1/606.1 kB 22.8 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7fee0abc-a87f-4baa-b256-3ff4968cddfb
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7fee0abc-a87f-4baa-b256-3ff4968cddfb
    Can't uninstall 

In [0]:
# Run this in a Databricks notebook
spark.sql("CREATE CATALOG IF NOT EXISTS main")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.financial")

# Table 1: raw filing registry
spark.sql("""
CREATE TABLE IF NOT EXISTS main.financial.raw_filings (
    ticker         STRING,
    quarter        STRING,
    filing_date    DATE,
    gcs_pdf_path   STRING,
    form_type      STRING,
    status         STRING,
    ingested_ts    TIMESTAMP
)
USING DELTA
""")

# Table 2: extracted KPIs
spark.sql("""
CREATE TABLE IF NOT EXISTS main.financial.extracted_kpis (
    ticker                    STRING,
    quarter                   STRING,
    revenue_usd_millions      DOUBLE,
    net_income_usd_millions   DOUBLE,
    eps_diluted               DOUBLE,
    gross_margin_pct          DOUBLE,
    operating_cash_flow       DOUBLE,
    revenue_yoy_growth_pct    DOUBLE,
    confidence_score          DOUBLE,
    validation_flags          STRING,
    hitl_required             BOOLEAN,
    reviewed                  BOOLEAN,
    pipeline_version          STRING,
    extraction_ts             TIMESTAMP
)
USING DELTA
""")

# Table 3: peer benchmarks
spark.sql("""
CREATE TABLE IF NOT EXISTS main.financial.peer_benchmarks (
    ticker                  STRING,
    quarter                 STRING,
    sector_median_margin    DOUBLE,
    relative_rank           INT,
    margin_delta_pct        DOUBLE,
    benchmark_ts            TIMESTAMP
)
USING DELTA
""")

# Table 4: investment briefs (final output)
spark.sql("""
CREATE TABLE IF NOT EXISTS main.financial.investment_briefs (
    ticker           STRING,
    quarter          STRING,
    brief_markdown   STRING,
    run_id           STRING,
    created_ts       TIMESTAMP
)
USING DELTA
""")

print("All Delta tables created successfully")

All Delta tables created successfully


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS main.financial.peer_discovery (
    ticker                  STRING,
    sub_industry            STRING,
    candidate_ticker        STRING,
    candidate_name          STRING,
    log_market_cap          DOUBLE,
    ebitda_margin           DOUBLE,
    roic                    DOUBLE,
    revenue_growth_1y       DOUBLE,
    net_debt_to_ebitda      DOUBLE,
    asset_turnover          DOUBLE,
    ev_to_revenue           DOUBLE,
    pe_ratio                DOUBLE,
    price_to_book           DOUBLE,
    sub_industry_match      DOUBLE,
    similarity_score        DOUBLE,
    is_selected_peer        BOOLEAN,
    cache_version           INTEGER,
    invalidation_reason     STRING,
    last_event_check_ts     TIMESTAMP,
    is_manually_valid       BOOLEAN,
    fetched_ts              TIMESTAMP
)
USING DELTA
""")

print("peer_discovery table created")

peer_discovery table created


In [0]:
%sql
ALTER TABLE main.financial.peer_benchmarks
ADD COLUMN peers_used STRING;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8097806631831192>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'ALTER TABLE main.financial.peer_benchmarks\nADD COLUMN peers_used STRING;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    

In [0]:
spark.sql("SHOW TABLES IN main.financial").show()

+---------+--------------------+-----------+
| database|           tableName|isTemporary|
+---------+--------------------+-----------+
|financial|      extracted_kpis|      false|
|financial|extracted_kpis_dr...|      false|
|financial|extracted_kpis_pr...|      false|
|financial|   investment_briefs|      false|
|financial|     peer_benchmarks|      false|
|financial|      peer_discovery|      false|
|financial|         raw_filings|      false|
+---------+--------------------+-----------+



In [0]:
display(
    spark.sql(
        "DESCRIBE TABLE main.financial.peer_benchmarks"
    )
)

col_name,data_type,comment
ticker,string,null
quarter,string,null
sector_median_margin,double,null
relative_rank,int,null
margin_delta_pct,double,null
benchmark_ts,timestamp,null
peers_used,string,null
